# Bulk RNA-seq WGCNA Template

用于 normalized expression 矩阵的共表达模块构建、模块-性状关联、hub gene 导出。建议输入 VST/rlog/log2(TPM+1) 后的表达矩阵，不要直接使用 raw counts。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./vsd_matrix.csv"       # genes x samples
TRAIT_FILE <- "./colData.csv"           # exported by RNAseq_General; rows are samples or contains SAMPLE_COLUMN
GENE_COLUMN <- NULL
SAMPLE_COLUMN <- "sample"
GROUP_COLUMN <- "condition"

MIN_MAD_QUANTILE <- 0.5                       # keep top variable genes by MAD
NETWORK_TYPE <- "signed"                      # "signed" recommended for biology
POWER_VECTOR <- c(1:10, seq(12, 30, 2))
MIN_MODULE_SIZE <- 30
MERGE_CUT_HEIGHT <- 0.25
TARGET_MODULES <- NULL                         # e.g. c("blue", "turquoise"); NULL = all significant modules
OUTDIR <- "RNAseq_WGCNA_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("WGCNA", "tidyverse", "pheatmap"))

suppressPackageStartupMessages({
  library(WGCNA)
  library(tidyverse)
  library(pheatmap)
})
allowWGCNAThreads()

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Expression and Traits

In [ ]:
expr_raw <- read.csv(EXPR_FILE, check.names = FALSE)
# Force the first column to character if GENE_COLUMN is not set, preventing numeric gene IDs.
if (is.null(GENE_COLUMN) && ncol(expr_raw) > 1) {
  expr_raw[[1]] <- as.character(expr_raw[[1]])
}
if (!is.null(GENE_COLUMN) && GENE_COLUMN %in% colnames(expr_raw)) {
  genes <- expr_raw[[GENE_COLUMN]]
  expr <- as.matrix(expr_raw[, setdiff(colnames(expr_raw), GENE_COLUMN), drop = FALSE])
  rownames(expr) <- genes
} else if (!is.numeric(expr_raw[[1]])) {
  genes <- expr_raw[[1]]
  expr <- as.matrix(expr_raw[, -1, drop = FALSE])
  rownames(expr) <- genes
} else {
  expr <- as.matrix(expr_raw)
}
mode(expr) <- "numeric"
expr <- expr[!duplicated(rownames(expr)) & rownames(expr) != "", , drop = FALSE]

traits <- read.csv(TRAIT_FILE, check.names = FALSE)
if (SAMPLE_COLUMN %in% colnames(traits)) {
  common_samples <- intersect(colnames(expr), traits[[SAMPLE_COLUMN]])
  expr <- expr[, common_samples, drop = FALSE]
  traits <- traits[match(common_samples, traits[[SAMPLE_COLUMN]]), ]
  rownames(traits) <- traits[[SAMPLE_COLUMN]]
} else {
  common_samples <- intersect(colnames(expr), rownames(traits))
  expr <- expr[, common_samples, drop = FALSE]
  traits <- traits[common_samples, , drop = FALSE]
}
stopifnot(all(colnames(expr) == rownames(traits)))
cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")

## 4. Gene Filtering and Sample QC

In [ ]:
gene_mad <- apply(expr, 1, mad, na.rm = TRUE)
expr <- expr[gene_mad >= quantile(gene_mad, MIN_MAD_QUANTILE, na.rm = TRUE), , drop = FALSE]
datExpr <- t(as.matrix(expr))

gsg <- goodSamplesGenes(datExpr, verbose = 3)
if (!gsg$allOK) {
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]
  traits <- traits[rownames(datExpr), , drop = FALSE]
}

sampleTree <- hclust(dist(datExpr), method = "average")
pdf(file.path(OUTDIR, "Sample_clustering.pdf"), width = 8, height = 5)
plot(sampleTree, main = "Sample clustering", sub = "", xlab = "")
dev.off()
cat("WGCNA input:", nrow(datExpr), "samples x", ncol(datExpr), "genes\n")

## 5. Soft Threshold Selection

In [ ]:
sft <- pickSoftThreshold(datExpr, powerVector = POWER_VECTOR, networkType = NETWORK_TYPE, verbose = 5)
soft_power <- sft$powerEstimate
if (is.na(soft_power)) {
  fit_df <- sft$fitIndices
  soft_power <- fit_df$Power[which.max(fit_df$SFT.R.sq)]
  message("No automatic power estimate; using max scale-free fit power: ", soft_power)
}

pdf(file.path(OUTDIR, "Soft_threshold_selection.pdf"), width = 10, height = 5)
par(mfrow = c(1, 2))
plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3]) * sft$fitIndices[,2], xlab = "Soft Threshold", ylab = "Scale Free Topology Model Fit", type = "n")
text(sft$fitIndices[,1], -sign(sft$fitIndices[,3]) * sft$fitIndices[,2], labels = POWER_VECTOR, col = "red")
abline(h = 0.8, col = "red")
plot(sft$fitIndices[,1], sft$fitIndices[,5], xlab = "Soft Threshold", ylab = "Mean Connectivity", type = "n")
text(sft$fitIndices[,1], sft$fitIndices[,5], labels = POWER_VECTOR, col = "red")
dev.off()
cat("Selected soft power:", soft_power, "\n")


## 6. Network Construction and Module Detection

In [ ]:
net <- blockwiseModules(
  datExpr,
  power = soft_power,
  networkType = NETWORK_TYPE,
  TOMType = NETWORK_TYPE,
  minModuleSize = MIN_MODULE_SIZE,
  reassignThreshold = 0,
  mergeCutHeight = MERGE_CUT_HEIGHT,
  numericLabels = TRUE,
  pamRespectsDendro = FALSE,
  saveTOMs = FALSE,
  verbose = 3
)
moduleColors <- labels2colors(net$colors)
MEs <- orderMEs(net$MEs)

write.csv(data.frame(gene = colnames(datExpr), module = moduleColors), file.path(OUTDIR, "WGCNA_gene_modules.csv"), row.names = FALSE)
saveRDS(list(net = net, moduleColors = moduleColors, MEs = MEs, datExpr = datExpr, traits = traits), file.path(OUTDIR, "WGCNA_network.rds"))

pdf(file.path(OUTDIR, "Module_dendrogram.pdf"), width = 10, height = 6)
plotDendroAndColors(net$dendrograms[[1]], moduleColors[net$blockGenes[[1]]], "Module colors", dendroLabels = FALSE, hang = 0.03, addGuide = TRUE)
dev.off()
print(table(moduleColors))


## 7. Module-Trait Correlation

In [ ]:
trait_model <- traits %>% select(-any_of(SAMPLE_COLUMN)) %>% select(where(~ is.numeric(.) || is.factor(.) || is.character(.)))
trait_model <- trait_model[, colSums(!is.na(trait_model)) > 0, drop = FALSE]
stopifnot(ncol(trait_model) > 0)
trait_numeric <- model.matrix(~ . - 1, data = trait_model)
moduleTraitCor <- cor(MEs, trait_numeric, use = "p")
moduleTraitP <- corPvalueStudent(moduleTraitCor, nrow(datExpr))

write.csv(moduleTraitCor, file.path(OUTDIR, "Module_trait_correlation.csv"))
write.csv(moduleTraitP, file.path(OUTDIR, "Module_trait_pvalue.csv"))

textMatrix <- paste(signif(moduleTraitCor, 2), "\n(", signif(moduleTraitP, 1), ")", sep = "")
pdf(file.path(OUTDIR, "Module_trait_heatmap.pdf"), width = max(8, ncol(trait_numeric) * 0.45), height = max(6, nrow(moduleTraitCor) * 0.28))
labeledHeatmap(Matrix = moduleTraitCor, xLabels = colnames(trait_numeric), yLabels = rownames(moduleTraitCor),
               ySymbols = rownames(moduleTraitCor), colorLabels = FALSE, colors = blueWhiteRed(50),
               textMatrix = textMatrix, setStdMargins = FALSE, cex.text = 0.7,
               zlim = c(-1, 1), main = "Module-trait relationships")
dev.off()


## 8. Hub Gene Export

In [ ]:
gene_module <- data.frame(gene = colnames(datExpr), module = moduleColors)
all_modules <- unique(moduleColors[moduleColors != "grey"])
if (!is.null(TARGET_MODULES)) all_modules <- intersect(all_modules, TARGET_MODULES)

hub_list <- list()
for (mod in all_modules) {
  mod_genes <- gene_module$gene[gene_module$module == mod]
  ME <- MEs[[paste0("ME", mod)]]
  kME <- cor(datExpr[, mod_genes, drop = FALSE], ME, use = "p")
  hub <- data.frame(gene = mod_genes, module = mod, kME = as.numeric(kME)) %>% arrange(desc(abs(kME)))
  hub_list[[mod]] <- hub
  write.csv(hub, file.path(OUTDIR, paste0("Hub_genes_", mod, ".csv")), row.names = FALSE)
}
write.csv(bind_rows(hub_list), file.path(OUTDIR, "Hub_genes_all_modules.csv"), row.names = FALSE)
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
